In [ ]:
from pathlib import Path
import sys
import importlib.util
if importlib.util.find_spec('duckdb') is None:
    raise ModuleNotFoundError('duckdb is missing from this Jupyter kernel; run %pip install -r requirements.txt, then restart the kernel.')
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'ypipe').is_dir(): PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT))
from ypipe import config

SEED=42
np.random.seed(SEED)
N_TOP_CATEGORIES=10
config.DATA.mkdir(parents=True,exist_ok=True)
_previous_con = globals().get('con')
if _previous_con is not None:
    try:
        _previous_con.close()
    except Exception:
        pass
class _NotebookQueryResult:
    def __init__(self, frame):
        self._frame = frame
    def df(self):
        return self._frame.copy()
    def fetchdf(self):
        return self._frame.copy()
    def fetchone(self):
        if self._frame.empty:
            return None
        return tuple(self._frame.iloc[0].tolist())
    def fetchall(self):
        return list(self._frame.itertuples(index=False, name=None))

class _NotebookReadOnlyDB:
    """Run each read query on a short-lived connection to avoid retaining a file lock."""
    def __init__(self, path):
        self.path = Path(path)
    def execute(self, sql, parameters=None):
        if not self.path.exists():
            if sql.strip().lower() == 'show tables':
                return _NotebookQueryResult(pd.DataFrame({'name': pd.Series(dtype='object')}))
            return _NotebookQueryResult(pd.DataFrame())
        db = duckdb.connect(str(self.path), read_only=True)
        try:
            result = db.execute(sql) if parameters is None else db.execute(sql, parameters)
            frame = result.fetchdf()
        finally:
            db.close()
        return _NotebookQueryResult(frame)
    def close(self):
        # Queries already close their own connection before returning.
        return None

con=_NotebookReadOnlyDB(config.DB_PATH)
TABLES=set(con.execute('SHOW TABLES').fetchdf()['name'])
print(f'Database: {config.DB_PATH} | seed={SEED}')

# Tổng kết và dashboard — Stage 6

## Câu hỏi
Pipeline đã kết nối dữ liệu Takeout, metadata YouTube, kho DuckDB/Parquet và các phân tích mô tả thành một luồng có thể kiểm tra như thế nào?

## Phương pháp
Chỉ đọc các bảng kết quả trong `warehouse.duckdb` và danh sách file Parquet. Dashboard tổng hợp số lượt mở, độ phủ metadata, thời gian, thể loại, tốc độ view hiện tại, cụm người dùng và cụm phiên. Không gọi pipeline/API và không hiển thị mã người dùng thật.

In [ ]:
def table_df(name):
    return con.execute(f'SELECT * FROM {name}').df() if name in TABLES else pd.DataFrame()
raw=table_df('watch_raw'); facts=table_df('video_facts'); uf=table_df('user_features')
sessions=table_df('sessions'); user_clusters=table_df('user_clusters'); session_clusters=table_df('session_clusters')
topics=table_df('topics'); ingested=table_df('ingested_files')
n_events=len(raw); n_users=raw.user_id.nunique() if len(raw) else 0
n_videos=raw.video_id.nunique() if len(raw) else 0
if len(raw):
    local_time=pd.to_datetime(raw.watched_at,utc=True).dt.tz_convert(config.TZ)
    date_range=f'{local_time.min():%d/%m/%Y} – {local_time.max():%d/%m/%Y}'
else: date_range='Chưa có dữ liệu'
if len(facts):
    status_counts=facts.metadata_status.value_counts(dropna=False)
    n_ok=int(status_counts.get('ok',0)); n_unavailable=int(status_counts.get('unavailable',0)); n_missing=int(status_counts.get('missing',0))
else: n_ok=n_unavailable=n_missing=0
parquet_files=sorted(config.PARQUET.glob('*.parquet')) if config.PARQUET.exists() else []
parquet_bytes=sum(p.stat().st_size for p in parquet_files)
db_bytes=config.DB_PATH.stat().st_size if config.DB_PATH.exists() else 0

summary=pd.DataFrame([
 {'Chỉ số':'Lượt mở trong Takeout','Giá trị':f'{n_events:,}'},
 {'Chỉ số':'Người tham gia','Giá trị':f'{n_users:,}'},
 {'Chỉ số':'Video duy nhất đã mở','Giá trị':f'{n_videos:,}'},
 {'Chỉ số':'Khoảng thời gian', 'Giá trị':date_range},
 {'Chỉ số':'Metadata ok / unavailable / missing','Giá trị':f'{n_ok:,} / {n_unavailable:,} / {n_missing:,} video'},
 {'Chỉ số':'File Takeout đã nạp','Giá trị':f'{len(ingested):,}'},
 {'Chỉ số':'Video facts / topic labels','Giá trị':f'{len(facts):,} / {len(topics):,}'},
 {'Chỉ số':'Phiên / cụm phiên','Giá trị':f'{len(sessions):,} / {len(session_clusters):,}'},
 {'Chỉ số':'File Parquet','Giá trị':f'{len(parquet_files):,} ({parquet_bytes/1024/1024:.1f} MiB)'},
 {'Chỉ số':'DuckDB','Giá trị':f'{db_bytes/1024/1024:.1f} MiB'},
])
display(summary)
print(f'Bảng trong DuckDB: {len(TABLES)} | bảng user clusters: {len(user_clusters)} dòng | session clusters: {len(session_clusters)} dòng')
print('Khoảng 500k sự kiện là mốc quy mô dự kiến của đồ án, không phải số liệu được suy ra nếu bảng hiện tại nhỏ hơn.')

## Kết quả: chỉ số và biểu đồ chính
Mỗi biểu đồ dùng số lượt mở hoặc video duy nhất theo đúng chú giải; panel hiệu suất dùng `views_per_day` đã log hóa để giảm lệch mạnh.

In [ ]:
if len(raw):
    monthly=raw.assign(month=pd.to_datetime(raw.watched_at,utc=True).dt.tz_convert(config.TZ).dt.to_period('M').astype(str))
    monthly=monthly.groupby('month').size()
else: monthly=pd.Series(dtype=int)
if len(facts):
    good=facts[facts.metadata_status.eq('ok')].copy()
    good['category_name']=good.category_name.fillna('Chưa phân loại')
    category_counts=good.groupby('category_name').agg(n_videos=('video_id','nunique'),n_events=('n_watch_events','sum')).sort_values('n_events',ascending=False)
else: good=pd.DataFrame(); category_counts=pd.DataFrame()

fig,axes=plt.subplots(2,2,figsize=(12,8))
if len(monthly):
    axes[0,0].bar(monthly.index,monthly.values,color='#2563eb'); axes[0,0].tick_params(axis='x',rotation=35)
else: axes[0,0].text(.5,.5,'Chưa có dữ liệu',ha='center',va='center',transform=axes[0,0].transAxes)
axes[0,0].set_title('Lượt mở theo tháng'); axes[0,0].set_ylabel('Số lượt mở'); axes[0,0].set_xlabel('Tháng')
if len(category_counts):
    top=category_counts.head(N_TOP_CATEGORIES).sort_values('n_events')
    axes[0,1].barh(top.index,top.n_events,color='#f59e0b')
else: axes[0,1].text(.5,.5,'Chưa có metadata thành công',ha='center',va='center',transform=axes[0,1].transAxes)
axes[0,1].set_title('Lượt mở theo thể loại'); axes[0,1].set_xlabel('Số lượt mở'); axes[0,1].set_ylabel('Thể loại')
if len(good) and good.views_per_day.notna().any():
    plot_good=good[good.views_per_day.notna()].copy(); plot_good['category_name']=plot_good.category_name.fillna('Chưa phân loại')
    order=plot_good.groupby('category_name').video_id.nunique().sort_values(ascending=False).head(6).index
    box_data=[np.log1p(plot_good.loc[plot_good.category_name==c,'views_per_day'].clip(lower=0)) for c in order]
    axes[1,0].boxplot(box_data,tick_labels=[str(c) for c in order],showfliers=False)
    axes[1,0].tick_params(axis='x',rotation=25)
else: axes[1,0].text(.5,.5,'Chưa có views/ngày',ha='center',va='center',transform=axes[1,0].transAxes)
axes[1,0].set_title('Hiệu suất snapshot theo thể loại'); axes[1,0].set_ylabel('log(1 + views/ngày)'); axes[1,0].set_xlabel('Thể loại')
if len(user_clusters) and 'cluster' in user_clusters:
    user_counts=user_clusters.groupby('cluster').size(); axes[1,1].bar([f'Cụm {int(x)+1}' for x in user_counts.index],user_counts.values,color='#7c3aed')
    axes[1,1].set_title('Số người trong từng cụm hành vi'); axes[1,1].set_ylabel('Số người'); axes[1,1].set_xlabel('Cụm')
elif len(session_clusters) and 'session_cluster' in session_clusters:
    session_counts=session_clusters.groupby('session_cluster').size(); axes[1,1].bar([f'Loại phiên {int(x)+1}' for x in session_counts.index],session_counts.values,color='#0f766e')
    axes[1,1].set_title('Số phiên trong từng loại'); axes[1,1].set_ylabel('Số phiên'); axes[1,1].set_xlabel('Loại phiên')
else: axes[1,1].text(.5,.5,'Chưa có bảng phân cụm',ha='center',va='center',transform=axes[1,1].transAxes)
fig.suptitle('Tổng quan lịch sử xem YouTube'); plt.tight_layout(); plt.show()

stage_map=pd.DataFrame([
 {'Stage':'1. Raw Data','Thành phần':'File Google Takeout JSON/ZIP được lưu trữ và theo dõi hash.'},
 {'Stage':'2. Ingestion','Thành phần':f"{len(ingested):,} file đã nạp; sự kiện được khử trùng lặp trong DuckDB."},
 {'Stage':'3. ETL / ELT','Thành phần':'Enrichment video/kênh qua YouTube Data API; comment tắt mặc định.'},
 {'Stage':'4. Storage','Thành phần':f'DuckDB + {len(parquet_files):,} snapshot Parquet.'},
 {'Stage':'5. Machine Learning','Thành phần':f'{len(topics):,} nhãn topic, {len(user_clusters):,} gán cụm user, {len(session_clusters):,} gán cụm phiên.'},
 {'Stage':'6. Visualization','Thành phần':'Notebook 00–06 và báo cáo HTML được tạo từ kho dữ liệu.'}
])
display(stage_map)
con.close()

## Insight
Dashboard nối đầu vào Takeout, enrichment API, storage DuckDB/Parquet, phân cụm và notebook. Khi sự kiện tăng lên hàng trăm nghìn, DuckDB/Parquet cho phép xử lý cục bộ theo cột; số liệu thật ở lần chạy này phải lấy từ bảng trên, không suy ra từ mục tiêu quy mô.

## Hạn chế
Mẫu chỉ phản ánh video người tham gia đã mở và còn truy xuất được metadata; video không khả dụng có thể gây lệch. `view_count` là snapshot tại `fetched_at`; `views_per_day` chỉ là chuẩn hóa đơn giản theo tuổi video. Phân cụm trên số người ít có thể thiếu ổn định, xem notebook 04 để đọc ARI. Thời gian 5–9 tháng không đủ kết luận mùa vụ năm. Các quan sát này mang tính mô tả, không chứng minh nhân quả.